# Week 6 — Predictive and Recommendation Analytics

**Project:** AI-Powered Career and Skill-Gap Recommendation System

This notebook implements the complete Week 6 pipeline:

1. Keyword baseline
2. TF-IDF recommendation
3. Sentence-Transformer semantic recommendation
4. Canadian labour-demand score
5. CIP-based education-alignment score
6. Demand-adjusted and education-adjusted variants
7. Final hybrid score = 35% skill-based candidate–occupation match + 35% semantic similarity + 20% Canadian labour demand + 10% CIP-based education alignment.
8. Model comparison
9. Example recommendations
10. Preliminary recommendation error analysis
11. Validation and CSV exports

The system is a **content-based recommendation/ranking system**, not a supervised career-classification model.


In [21]:
# ============================================================
# 0. CONFIGURATION AND LIBRARIES
# ============================================================

import os
import re
import glob
import warnings
import subprocess
import sys
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score

warnings.filterwarnings("ignore")

PROJECT_PATH = r"C:\Users\Admin\Capstone_Project"
TABLES_PATH = os.path.join(PROJECT_PATH, "Outputs", "Tables")
WEEK5_PATH = os.path.join(TABLES_PATH, "Week5")
WEEK6_PATH = os.path.join(TABLES_PATH, "Week6")
DATA_PATH = os.path.join(PROJECT_PATH, "Data")
RAW_DATA_PATH = os.path.join(DATA_PATH, "Raw_Data")

os.makedirs(WEEK6_PATH, exist_ok=True)

print("=" * 80)
print("WEEK 6 — PREDICTIVE AND RECOMMENDATION ANALYTICS")
print("=" * 80)
print("Project:", "AI-Powered Career and Skill-Gap Recommendation System")
print("Project path:", PROJECT_PATH)
print("Week 5 path exists:", os.path.exists(WEEK5_PATH))
print("Week 6 output path:", WEEK6_PATH)

WEEK 6 — PREDICTIVE AND RECOMMENDATION ANALYTICS
Project: AI-Powered Career and Skill-Gap Recommendation System
Project path: C:\Users\Admin\Capstone_Project
Week 5 path exists: True
Week 6 output path: C:\Users\Admin\Capstone_Project\Outputs\Tables\Week6


In [23]:
# ============================================================
# 1. LOAD WEEK 5 INPUTS
# ============================================================

def load_csv(filename, folder=WEEK5_PATH):
    path = os.path.join(folder, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file not found: {path}")
    return pd.read_csv(path)

# Week 5 candidate files
candidate_features = load_csv(
    "week5_candidate_features.csv",
    folder=WEEK5_PATH
)

candidate_occupation_scores = load_csv(
    "week5_candidate_occupation_scores.csv",
    folder=WEEK5_PATH
)

candidate_onet_skill_matrix = load_csv(
    "week5_candidate_onet_skill_matrix.csv",
    folder=WEEK5_PATH
)

# O*NET integrated and detailed profiles
onet_integrated_profile = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_integrated_occupation_profile.csv"
    )
)

onet_cip_integrated_profile = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_cip_integrated_occupation_profile.csv"
    )
)

onet_skills_profile = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_skills_profile.csv"
    )
)

onet_knowledge_profile = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_knowledge_profile.csv"
    )
)

onet_abilities_profile = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_abilities_profile.csv"
    )
)

onet_tasks_profile = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_tasks_profile.csv"
    )
)

onet_education_profile = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_education_profile.csv"
    )
)

onet_training_profile = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_training_profile.csv"
    )
)

print("Candidate features:", candidate_features.shape)
print("Baseline scores:", candidate_occupation_scores.shape)
print("Candidate-O*NET skill matrix:", candidate_onet_skill_matrix.shape)
print("O*NET integrated profile:", onet_integrated_profile.shape)
print("O*NET + CIP integrated profile:", onet_cip_integrated_profile.shape)
print("O*NET skills:", onet_skills_profile.shape)
print("O*NET knowledge:", onet_knowledge_profile.shape)
print("O*NET abilities:", onet_abilities_profile.shape)
print("O*NET tasks:", onet_tasks_profile.shape)
print("O*NET education:", onet_education_profile.shape)
print("O*NET training:", onet_training_profile.shape)

assert candidate_features["candidate_id"].nunique() == 63
assert candidate_occupation_scores["candidate_id"].nunique() == 63

Candidate features: (63, 16)
Baseline scores: (945, 5)
Candidate-O*NET skill matrix: (63, 23)
O*NET integrated profile: (20, 10)
O*NET + CIP integrated profile: (20, 12)
O*NET skills: (200, 7)
O*NET knowledge: (660, 7)
O*NET abilities: (1040, 7)
O*NET tasks: (487, 7)
O*NET education: (300, 9)
O*NET training: (592, 18)


In [25]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def combine_unique_text(series):
    values = (
        series.dropna()
        .astype(str)
        .str.strip()
    )
    values = values[values.ne("")]
    return " | ".join(pd.unique(values))

def normalize_per_candidate(df, score_col, output_col):
    df = df.copy()
    df[output_col] = (
        df.groupby("candidate_id")[score_col]
        .transform(
            lambda x: (x - x.min()) / (x.max() - x.min())
            if x.max() != x.min()
            else 1.0
        )
    )
    return df

def max_normalize(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    maximum = series.max()
    if maximum == 0:
        return pd.Series(0.0, index=series.index)
    return series / maximum

def clean_text(x):
    x = "" if pd.isna(x) else str(x)
    x = re.sub(r"[^a-zA-Z0-9+#.\-/ ]+", " ", x.lower())
    x = re.sub(r"\s+", " ", x).strip()
    return x

def token_set(x):
    return set(clean_text(x).split())

def find_column(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    for col in df.columns:
        low = col.lower()
        if any(c.lower() in low for c in candidates):
            return col
    if required:
        raise KeyError(
            f"Could not identify a column from {candidates}. "
            f"Available columns: {df.columns.tolist()}"
        )
    return None

In [27]:
# ============================================================
# 3. BUILD OCCUPATION DOCUMENTS
# ============================================================

occupation_skills_text = (
    onet_skills_profile.groupby("selected_occupation")["skill"]
    .apply(combine_unique_text)
    .reset_index(name="skills_text")
)

occupation_knowledge_text = (
    onet_knowledge_profile.groupby("selected_occupation")["knowledge_area"]
    .apply(combine_unique_text)
    .reset_index(name="knowledge_text")
)

occupation_abilities_text = (
    onet_abilities_profile.groupby("selected_occupation")["ability"]
    .apply(combine_unique_text)
    .reset_index(name="abilities_text")
)

occupation_tasks_text = (
    onet_tasks_profile.groupby("selected_occupation")["task"]
    .apply(combine_unique_text)
    .reset_index(name="tasks_text")
)

occupation_education_text = (
    onet_education_profile.groupby("selected_occupation")["education_level"]
    .apply(combine_unique_text)
    .reset_index(name="education_text")
)

occupation_training_text = (
    onet_training_profile.groupby("selected_occupation")["Element Name"]
    .apply(combine_unique_text)
    .reset_index(name="training_text")
)

occupation_titles = (
    onet_integrated_profile[["selected_occupation", "onet_title"]]
    .groupby("selected_occupation")["onet_title"]
    .apply(combine_unique_text)
    .reset_index()
)

occupation_documents = occupation_titles.copy()

for part in [
    occupation_skills_text,
    occupation_knowledge_text,
    occupation_abilities_text,
    occupation_tasks_text,
    occupation_education_text,
    occupation_training_text
]:
    occupation_documents = occupation_documents.merge(
        part, on="selected_occupation", how="left"
    )

text_cols = [
    "onet_title", "skills_text", "knowledge_text", "abilities_text",
    "tasks_text", "education_text", "training_text"
]
occupation_documents[text_cols] = occupation_documents[text_cols].fillna("")

occupation_documents["occupation_document"] = (
    "Occupation: " + occupation_documents["selected_occupation"] +
    " O*NET Title: " + occupation_documents["onet_title"] +
    " Skills: " + occupation_documents["skills_text"] +
    " Knowledge: " + occupation_documents["knowledge_text"] +
    " Abilities: " + occupation_documents["abilities_text"] +
    " Tasks: " + occupation_documents["tasks_text"] +
    " Education: " + occupation_documents["education_text"] +
    " Training: " + occupation_documents["training_text"]
)

print("Occupation documents:", occupation_documents.shape)
print("Unique occupations:", occupation_documents["selected_occupation"].nunique())

assert occupation_documents["selected_occupation"].nunique() == 15
assert occupation_documents["selected_occupation"].duplicated().sum() == 0

Occupation documents: (15, 9)
Unique occupations: 15


In [29]:
# ============================================================
# 4. BUILD CANDIDATE PROFILE DOCUMENTS
# ============================================================

candidate_text_columns = [
    "technical_skills", "technologies", "databases",
    "software_tools", "professional_skills", "certifications",
    "degree_level", "field_of_study", "raw_skills"
]

for col in candidate_text_columns:
    candidate_features[col] = (
        candidate_features[col].fillna("").astype(str).str.strip()
    )

candidate_features["candidate_document"] = (
    "Technical Skills: " + candidate_features["technical_skills"] +
    " Technologies: " + candidate_features["technologies"] +
    " Databases: " + candidate_features["databases"] +
    " Software Tools: " + candidate_features["software_tools"] +
    " Professional Skills: " + candidate_features["professional_skills"] +
    " Certifications: " + candidate_features["certifications"] +
    " Degree Level: " + candidate_features["degree_level"] +
    " Field of Study: " + candidate_features["field_of_study"] +
    " Additional Skills: " + candidate_features["raw_skills"]
)

assert candidate_features["candidate_id"].nunique() == 63
assert candidate_features["candidate_document"].isna().sum() == 0

print("Candidate documents created:", candidate_features.shape)

Candidate documents created: (63, 17)


**Model 1** - Keyword Baseline

A simple and transparent benchmark for occupation recommendation.

Combines each candidate's skills and technical profile.
Compares candidate tokens with each occupation's skills, knowledge, and abilities.
Counts matching tokens and calculates a keyword similarity score.
Normalizes scores per candidate and ranks the 15 occupations.

Result: 63 candidates × 15 occupations = 945 candidate–occupation scores.

In [31]:
# ============================================================
# 5. KEYWORD BASELINE
# ============================================================

# A transparent baseline based on token overlap between candidate
# skills/profile and occupation skill/knowledge/ability terms.

candidate_baseline_text = (
    candidate_features["technical_skills"].fillna("") + " " +
    candidate_features["technologies"].fillna("") + " " +
    candidate_features["databases"].fillna("") + " " +
    candidate_features["software_tools"].fillna("") + " " +
    candidate_features["professional_skills"].fillna("") + " " +
    candidate_features["raw_skills"].fillna("")
)

occupation_baseline_text = (
    occupation_documents["skills_text"].fillna("") + " " +
    occupation_documents["knowledge_text"].fillna("") + " " +
    occupation_documents["abilities_text"].fillna("")
)

keyword_rows = []

for c_idx, candidate_id in enumerate(candidate_features["candidate_id"]):
    c_tokens = token_set(candidate_baseline_text.iloc[c_idx])

    for o_idx, occupation in enumerate(occupation_documents["selected_occupation"]):
        o_tokens = token_set(occupation_baseline_text.iloc[o_idx])

        if not c_tokens or not o_tokens:
            score = 0.0
            matched = 0
        else:
            matched = len(c_tokens & o_tokens)
            score = matched / len(c_tokens)

        keyword_rows.append({
            "candidate_id": candidate_id,
            "selected_occupation": occupation,
            "keyword_match_count": matched,
            "keyword_similarity_score": score
        })

keyword_recommendations = pd.DataFrame(keyword_rows)
keyword_recommendations = normalize_per_candidate(
    keyword_recommendations,
    "keyword_similarity_score",
    "keyword_normalized_score"
)
keyword_recommendations["keyword_percentage"] = (
    keyword_recommendations["keyword_normalized_score"] * 100
).round(2)
keyword_recommendations["keyword_rank"] = (
    keyword_recommendations.groupby("candidate_id")["keyword_normalized_score"]
    .rank(method="first", ascending=False).astype(int)
)

print("Keyword baseline:", keyword_recommendations.shape)
assert len(keyword_recommendations) == 63 * 15
assert keyword_recommendations.duplicated(
    ["candidate_id", "selected_occupation"]
).sum() == 0

Keyword baseline: (945, 7)


**Model 2** — TF-IDF Recommendation Model
(Term Frequency–Inverse Document Frequency)

This model measures how closely each candidate's profile matches each occupation based on the importance of words and phrases.

Converts candidate and occupation documents into TF-IDF numerical representations.  
Considers both individual words and two-word phrases.  
Gives higher importance to terms that are more specific and less common across all documents.  
Uses cosine similarity to measure candidate–occupation similarity.  
Normalizes the scores per candidate and ranks the 15 occupations.  

Result: 63 candidates × 15 occupations = 945 TF-IDF similarity scores.

In [33]:
# ============================================================
# 6. TF-IDF RECOMMENDATION MODEL
# ============================================================

candidate_docs = candidate_features["candidate_document"].tolist()
occupation_docs = occupation_documents["occupation_document"].tolist()

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = tfidf_vectorizer.fit_transform(candidate_docs + occupation_docs)

candidate_tfidf = tfidf_matrix[:len(candidate_docs)]
occupation_tfidf = tfidf_matrix[len(candidate_docs):]

tfidf_similarity = cosine_similarity(candidate_tfidf, occupation_tfidf)

tfidf_rows = []
for i, candidate_id in enumerate(candidate_features["candidate_id"]):
    for j, occupation in enumerate(occupation_documents["selected_occupation"]):
        score = float(tfidf_similarity[i, j])
        tfidf_rows.append({
            "candidate_id": candidate_id,
            "selected_occupation": occupation,
            "tfidf_similarity_score": score
        })

tfidf_recommendations = pd.DataFrame(tfidf_rows)
tfidf_recommendations = normalize_per_candidate(
    tfidf_recommendations,
    "tfidf_similarity_score",
    "tfidf_normalized_score"
)
tfidf_recommendations["tfidf_percentage"] = (
    tfidf_recommendations["tfidf_normalized_score"] * 100
).round(2)
tfidf_recommendations["tfidf_rank"] = (
    tfidf_recommendations.groupby("candidate_id")["tfidf_similarity_score"]
    .rank(method="first", ascending=False).astype(int)
)

print("TF-IDF vocabulary:", len(tfidf_vectorizer.get_feature_names_out()))
print("TF-IDF results:", tfidf_recommendations.shape)

assert len(tfidf_recommendations) == 945
assert tfidf_recommendations["tfidf_similarity_score"].isna().sum() == 0

TF-IDF vocabulary: 6161
TF-IDF results: (945, 6)


**Model 3** — Sentence-Transformer Semantic Model

This model measures semantic similarity, allowing the system to identify similar meaning even when exact words do not match.

Uses the Sentence-Transformer all-MiniLM-L6-v2 model.  
Converts candidate and occupation profiles into embeddings (numerical representations of meaning).  
Calculates cosine similarity between each candidate and occupation embedding.  
Normalizes the scores per candidate and ranks the 15 occupations.  

Result: 63 candidates × 15 occupations = 945 semantic similarity scores.  

This model goes beyond exact keyword matching by considering the meaning and context of the profiles.

In [35]:
# ============================================================
# 7. SENTENCE-TRANSFORMER SEMANTIC MODEL
# ============================================================

# If needed, run this once in a notebook:
# %pip install sentence-transformers

from sentence_transformers import SentenceTransformer

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

candidate_embeddings = semantic_model.encode(
    candidate_docs,
    convert_to_numpy=True,
    show_progress_bar=True
)

occupation_embeddings = semantic_model.encode(
    occupation_docs,
    convert_to_numpy=True,
    show_progress_bar=True
)

semantic_similarity = cosine_similarity(
    candidate_embeddings,
    occupation_embeddings
)

semantic_rows = []
for i, candidate_id in enumerate(candidate_features["candidate_id"]):
    for j, occupation in enumerate(occupation_documents["selected_occupation"]):
        score = float(semantic_similarity[i, j])
        semantic_rows.append({
            "candidate_id": candidate_id,
            "selected_occupation": occupation,
            "semantic_similarity_score": score
        })

semantic_recommendations = pd.DataFrame(semantic_rows)
semantic_recommendations = normalize_per_candidate(
    semantic_recommendations,
    "semantic_similarity_score",
    "semantic_normalized_score"
)
semantic_recommendations["semantic_percentage"] = (
    semantic_recommendations["semantic_normalized_score"] * 100
).round(2)
semantic_recommendations["semantic_rank"] = (
    semantic_recommendations.groupby("candidate_id")["semantic_similarity_score"]
    .rank(method="first", ascending=False).astype(int)
)

print("Semantic results:", semantic_recommendations.shape)
assert len(semantic_recommendations) == 945
assert semantic_recommendations["semantic_similarity_score"].isna().sum() == 0

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Semantic results: (945, 6)


In [39]:
# ============================================================
# 8. CANADIAN LABOUR DEMAND SCORE
# ============================================================

# The existing Week 6 notebook already creates this file from the
# June 2026 Job Bank open-data dataset using:
# 40% posting volume + 40% vacancy volume + 20% geographic spread.
#
# If the file does not exist, run the Job Bank construction section
# from the original Week 6 notebook first.

demand_file = os.path.join(
    WEEK6_PATH,
    "week6_canadian_labour_demand_scores.csv"
)

if not os.path.exists(demand_file):
    raise FileNotFoundError(
        "week6_canadian_labour_demand_scores.csv was not found. "
        "Run the Job Bank demand-building section from Week 6 first."
    )

occupation_demand = pd.read_csv(demand_file)

required_demand_cols = [
    "selected_occupation", "posting_count",
    "total_vacancies", "province_count",
    "demand_score", "demand_percentage"
]

missing = [c for c in required_demand_cols if c not in occupation_demand.columns]
if missing:
    raise KeyError(f"Demand file is missing columns: {missing}")

occupation_demand = occupation_demand[
    required_demand_cols
].drop_duplicates("selected_occupation")

occupation_demand["demand_score"] = pd.to_numeric(
    occupation_demand["demand_score"], errors="coerce"
).fillna(0).clip(0, 1)

print("Demand occupations:", occupation_demand["selected_occupation"].nunique())

assert occupation_demand["selected_occupation"].nunique() == 15
assert occupation_demand["demand_score"].isna().sum() == 0

Demand occupations: 15


In [41]:
# ============================================================
# 9. CIP-BASED EDUCATION ALIGNMENT
# ============================================================

# Preferred source: Week 5 CIP scoring trace / candidate-CIP mapping.
# The code below tries to use an existing candidate-occupation CIP score
# if Week 5 produced one. Otherwise it uses candidate CIP codes plus an
# occupation CIP-code mapping if those columns are available.

cip_mapping_path = os.path.join(WEEK5_PATH, "week5_candidate_cip_mapping.csv")
cip_trace_path = os.path.join(WEEK5_PATH, "week5_cip_scoring_trace.csv")
cip_summary_path = os.path.join(WEEK5_PATH, "week5_final_cip_summary.csv")

education_alignment = None

# ---------- Strategy A: existing candidate-occupation CIP score ----------
if os.path.exists(cip_trace_path):
    trace = pd.read_csv(cip_trace_path)

    if {"candidate_id", "selected_occupation"}.issubset(trace.columns):
        score_candidates = [
            c for c in trace.columns
            if "score" in c.lower() and "cip" in c.lower()
        ]
        if not score_candidates:
            score_candidates = [
                c for c in trace.columns
                if "alignment" in c.lower()
            ]

        if score_candidates:
            score_col = score_candidates[0]
            tmp = trace[
                ["candidate_id", "selected_occupation", score_col]
            ].copy()
            tmp[score_col] = pd.to_numeric(tmp[score_col], errors="coerce").fillna(0)

            education_alignment = tmp.rename(
                columns={score_col: "education_raw_score"}
            )

# ---------- Strategy B: candidate CIP + occupation CIP relationship ----------
if education_alignment is None and os.path.exists(cip_mapping_path):
    candidate_cip = pd.read_csv(cip_mapping_path)

    # Candidate ID
    if "candidate_id" in candidate_cip.columns:
        cip_code_candidates = [
            c for c in candidate_cip.columns
            if "cip" in c.lower() and ("code" in c.lower() or "id" in c.lower())
        ]

        if cip_code_candidates:
            candidate_cip_col = cip_code_candidates[0]

            # Look for an occupation-to-CIP relationship in available outputs.
            # The mapping may already contain occupation + CIP.
            occ_col = next(
                (c for c in candidate_cip.columns
                 if "occupation" in c.lower() or "selected_occupation" in c.lower()),
                None
            )

            if occ_col:
                tmp = candidate_cip[
                    ["candidate_id", occ_col, candidate_cip_col]
                ].copy()
                tmp.columns = ["candidate_id", "selected_occupation", "cip_code"]

                # If the same file already gives candidate-occupation-CIP relationships,
                # use a binary alignment score.
                tmp["cip_code"] = tmp["cip_code"].astype(str).str.extract(
                    r"(\d{2}\.\d{4})", expand=False
                )
                tmp["education_raw_score"] = tmp["cip_code"].notna().astype(float)

                education_alignment = (
                    tmp.groupby(["candidate_id", "selected_occupation"])
                    ["education_raw_score"]
                    .max()
                    .reset_index()
                )

# ---------- Strategy C: CIP summary score, if it is already candidate-specific ----------
if education_alignment is None and os.path.exists(cip_summary_path):
    summary = pd.read_csv(cip_summary_path)
    if {"candidate_id", "selected_occupation"}.issubset(summary.columns):
        score_candidates = [
            c for c in summary.columns
            if "score" in c.lower() or "alignment" in c.lower()
        ]
        if score_candidates:
            score_col = score_candidates[0]
            education_alignment = summary[
                ["candidate_id", "selected_occupation", score_col]
            ].rename(columns={score_col: "education_raw_score"})
            education_alignment["education_raw_score"] = pd.to_numeric(
                education_alignment["education_raw_score"], errors="coerce"
            ).fillna(0)

if education_alignment is None:
    raise RuntimeError(
        "A true CIP-based education-alignment score could not be constructed "
        "from the available Week 5 files. Please inspect the columns in "
        "week5_candidate_cip_mapping.csv or week5_cip_scoring_trace.csv and "
        "map candidate CIP codes to occupation CIP pathways. Do not replace "
        "this with an arbitrary education score."
    )

# Normalize education score globally to 0–1.
education_alignment["education_raw_score"] = pd.to_numeric(
    education_alignment["education_raw_score"], errors="coerce"
).fillna(0)

max_edu = education_alignment["education_raw_score"].max()
if max_edu > 0:
    education_alignment["education_score"] = (
        education_alignment["education_raw_score"] / max_edu
    )
else:
    education_alignment["education_score"] = 0.0

education_alignment["education_percentage"] = (
    education_alignment["education_score"] * 100
).round(2)

education_alignment = (
    education_alignment[
        ["candidate_id", "selected_occupation",
         "education_score", "education_percentage"]
    ]
    .drop_duplicates(["candidate_id", "selected_occupation"])
)

print("Education alignment rows:", len(education_alignment))
print("Candidates covered:", education_alignment["candidate_id"].nunique())

Education alignment rows: 315
Candidates covered: 63


In [43]:
# ============================================================
# 10. BUILD A COMPLETE 63 × 15 EDUCATION MATRIX
# ============================================================

all_pairs = pd.MultiIndex.from_product(
    [
        candidate_features["candidate_id"].unique(),
        occupation_documents["selected_occupation"].unique()
    ],
    names=["candidate_id", "selected_occupation"]
).to_frame(index=False)

education_alignment = all_pairs.merge(
    education_alignment,
    on=["candidate_id", "selected_occupation"],
    how="left"
)

education_alignment["education_score"] = (
    education_alignment["education_score"].fillna(0).clip(0, 1)
)

education_alignment["education_percentage"] = (
    education_alignment["education_score"] * 100
).round(2)

print("Complete education matrix:", education_alignment.shape)
print("Missing education scores:", education_alignment["education_score"].isna().sum())

assert len(education_alignment) == 945
assert education_alignment["education_score"].isna().sum() == 0

Complete education matrix: (945, 4)
Missing education scores: 0


In [45]:
# ============================================================
# 11. MERGE ALL MODEL COMPONENTS
# ============================================================

skill_scores = candidate_occupation_scores[
    ["candidate_id", "selected_occupation", "recommendation_score",
     "recommendation_percentage"]
].copy()

skill_scores = normalize_per_candidate(
    skill_scores,
    "recommendation_score",
    "skill_normalized_score"
)
skill_scores["skill_percentage"] = (
    skill_scores["skill_normalized_score"] * 100
).round(2)

components = (
    skill_scores[
        ["candidate_id", "selected_occupation",
         "recommendation_score", "recommendation_percentage",
         "skill_normalized_score", "skill_percentage"]
    ]
    .merge(
        tfidf_recommendations[
            ["candidate_id", "selected_occupation",
             "tfidf_similarity_score", "tfidf_normalized_score",
             "tfidf_percentage"]
        ],
        on=["candidate_id", "selected_occupation"], how="inner"
    )
    .merge(
        semantic_recommendations[
            ["candidate_id", "selected_occupation",
             "semantic_similarity_score", "semantic_normalized_score",
             "semantic_percentage"]
        ],
        on=["candidate_id", "selected_occupation"], how="inner"
    )
    .merge(
        occupation_demand,
        on="selected_occupation", how="left"
    )
    .merge(
        education_alignment,
        on=["candidate_id", "selected_occupation"], how="left"
    )
)

components["demand_score"] = components["demand_score"].fillna(0).clip(0, 1)
components["demand_percentage"] = (
    components["demand_score"] * 100
).round(2)

components["education_score"] = components["education_score"].fillna(0).clip(0, 1)
components["education_percentage"] = (
    components["education_score"] * 100
).round(2)

print("Merged component matrix:", components.shape)

assert len(components) == 945
assert components.duplicated(
    ["candidate_id", "selected_occupation"]
).sum() == 0

Merged component matrix: (945, 19)


In [47]:
# ============================================================
# 12. TEST THE FINAL WEIGHTS
# ============================================================

# Candidate–occupation match = 70%
#   35% O*NET/Week-5 skill match
#   35% semantic similarity
# Canadian labour demand = 20%
# Education pathway = 10%

SKILL_WEIGHT = 0.35
SEMANTIC_WEIGHT = 0.35
DEMAND_WEIGHT = 0.20
EDUCATION_WEIGHT = 0.10

assert abs(
    SKILL_WEIGHT +
    SEMANTIC_WEIGHT +
    DEMAND_WEIGHT +
    EDUCATION_WEIGHT - 1.0
) < 1e-9

components["candidate_match_score"] = (
    SKILL_WEIGHT * components["skill_normalized_score"] +
    SEMANTIC_WEIGHT * components["semantic_normalized_score"]
)

components["final_hybrid_score"] = (
    components["candidate_match_score"] +
    DEMAND_WEIGHT * components["demand_score"] +
    EDUCATION_WEIGHT * components["education_score"]
)

components["final_hybrid_percentage"] = (
    components["final_hybrid_score"] * 100
).round(2)

components["final_hybrid_rank"] = (
    components.groupby("candidate_id")["final_hybrid_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

components = components.sort_values(
    ["candidate_id", "final_hybrid_rank"]
).reset_index(drop=True)

print("Final hybrid model created.")
print("Weights:")
print("Skill:", SKILL_WEIGHT)
print("Semantic:", SEMANTIC_WEIGHT)
print("Demand:", DEMAND_WEIGHT)
print("Education:", EDUCATION_WEIGHT)

assert components["final_hybrid_score"].isna().sum() == 0
assert components["final_hybrid_score"].between(0, 1).all()

Final hybrid model created.
Weights:
Skill: 0.35
Semantic: 0.35
Demand: 0.2
Education: 0.1


In [49]:
# ============================================================
# 13. DEMAND-ADJUSTED AND EDUCATION-ADJUSTED MODELS
# ============================================================

# Demand-adjusted: 80% candidate match + 20% demand
components["demand_adjusted_score"] = (
    0.40 * components["skill_normalized_score"] +
    0.40 * components["semantic_normalized_score"] +
    0.20 * components["demand_score"]
)

components["demand_adjusted_rank"] = (
    components.groupby("candidate_id")["demand_adjusted_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Education-adjusted: 90% candidate match + 10% education
components["education_adjusted_score"] = (
    0.45 * components["skill_normalized_score"] +
    0.45 * components["semantic_normalized_score"] +
    0.10 * components["education_score"]
)

components["education_adjusted_rank"] = (
    components.groupby("candidate_id")["education_adjusted_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# TF-IDF + demand and TF-IDF + education variants
components["tfidf_demand_score"] = (
    0.80 * components["tfidf_normalized_score"] +
    0.20 * components["demand_score"]
)
components["tfidf_demand_rank"] = (
    components.groupby("candidate_id")["tfidf_demand_score"]
    .rank(method="first", ascending=False).astype(int)
)

components["tfidf_education_score"] = (
    0.90 * components["tfidf_normalized_score"] +
    0.10 * components["education_score"]
)
components["tfidf_education_rank"] = (
    components.groupby("candidate_id")["tfidf_education_score"]
    .rank(method="first", ascending=False).astype(int)
)

print("Adjusted recommendation models created.")

Adjusted recommendation models created.


In [51]:
# ============================================================
# 14. MODEL COMPARISON TABLE
# ============================================================

def top1_table(df, score_col, rank_col, model_name):
    top1 = df[df[rank_col] == 1][
        ["candidate_id", "selected_occupation", score_col]
    ].copy()
    top1["model"] = model_name
    return top1

model_top1 = pd.concat([
    top1_table(components, "keyword_score" if "keyword_score" in components else "skill_normalized_score",
               "keyword_rank" if "keyword_rank" in components else "final_hybrid_rank",
               "Skill Baseline"),
    top1_table(
        tfidf_recommendations.rename(columns={"tfidf_normalized_score":"score"}),
        "score", "tfidf_rank", "TF-IDF"
    ),
    top1_table(
        semantic_recommendations.rename(columns={"semantic_normalized_score":"score"}),
        "score", "semantic_rank", "Sentence Transformer"
    ),
    top1_table(
        components, "demand_adjusted_score", "demand_adjusted_rank",
        "Demand-Adjusted"
    ),
    top1_table(
        components, "education_adjusted_score", "education_adjusted_rank",
        "Education-Adjusted"
    ),
    top1_table(
        components, "final_hybrid_score", "final_hybrid_rank",
        "Final Hybrid"
    )
], ignore_index=True)

# The keyword baseline is kept separately because its columns are not
# part of the component matrix.
keyword_top1 = keyword_recommendations[
    keyword_recommendations["keyword_rank"] == 1
][["candidate_id", "selected_occupation", "keyword_normalized_score"]].copy()
keyword_top1 = keyword_top1.rename(
    columns={"keyword_normalized_score": "score"}
)
keyword_top1["model"] = "Keyword Baseline"

tfidf_top1 = tfidf_recommendations[
    tfidf_recommendations["tfidf_rank"] == 1
][["candidate_id", "selected_occupation", "tfidf_normalized_score"]].copy()
tfidf_top1 = tfidf_top1.rename(
    columns={"tfidf_normalized_score": "score"}
)
tfidf_top1["model"] = "TF-IDF"

semantic_top1 = semantic_recommendations[
    semantic_recommendations["semantic_rank"] == 1
][["candidate_id", "selected_occupation", "semantic_normalized_score"]].copy()
semantic_top1 = semantic_top1.rename(
    columns={"semantic_normalized_score": "score"}
)
semantic_top1["model"] = "Sentence Transformer"

demand_top1 = components[
    components["demand_adjusted_rank"] == 1
][["candidate_id", "selected_occupation", "demand_adjusted_score"]].copy()
demand_top1 = demand_top1.rename(columns={"demand_adjusted_score":"score"})
demand_top1["model"] = "Demand-Adjusted"

education_top1 = components[
    components["education_adjusted_rank"] == 1
][["candidate_id", "selected_occupation", "education_adjusted_score"]].copy()
education_top1 = education_top1.rename(columns={"education_adjusted_score":"score"})
education_top1["model"] = "Education-Adjusted"

hybrid_top1 = components[
    components["final_hybrid_rank"] == 1
][["candidate_id", "selected_occupation", "final_hybrid_score"]].copy()
hybrid_top1 = hybrid_top1.rename(columns={"final_hybrid_score":"score"})
hybrid_top1["model"] = "Final Hybrid"

model_top1 = pd.concat(
    [
        keyword_top1, tfidf_top1, semantic_top1,
        demand_top1, education_top1, hybrid_top1
    ],
    ignore_index=True
)

model_summary = (
    model_top1.groupby("model")
    .agg(
        candidates=("candidate_id", "nunique"),
        mean_top1_score=("score", "mean"),
        median_top1_score=("score", "median")
    )
    .reset_index()
)

print("Model comparison:")
display(model_summary)

assert set(model_summary["model"]) == {
    "Keyword Baseline",
    "TF-IDF",
    "Sentence Transformer",
    "Demand-Adjusted",
    "Education-Adjusted",
    "Final Hybrid"
}

Model comparison:


,model,candidates,mean_top1_score,median_top1_score
0,Demand-Adjusted,63,0.769027,0.773408
1,Education-Adjusted,63,0.887806,0.918539
2,Final Hybrid,63,0.781823,0.782459
3,Keyword Baseline,63,1.000000,1.000000
4,Sentence Transformer,63,1.000000,1.000000
5,TF-IDF,63,1.000000,1.000000


In [55]:
# ============================================================
# 15. TOP-5 AND TOP-1 FINAL RECOMMENDATIONS
# ============================================================

final_top5 = components[
    components["final_hybrid_rank"] <= 5
].copy()

final_top1 = components[
    components["final_hybrid_rank"] == 1
].copy()

print("Final Top-5 rows:", len(final_top5))
print("Final Top-1 rows:", len(final_top1))

assert len(final_top5) == 63 * 5
assert len(final_top1) == 63
assert final_top1["candidate_id"].nunique() == 63

print("\nExample final recommendations:")
display(
    final_top5[
        [
            "candidate_id",
            "final_hybrid_rank",
            "selected_occupation",
            "final_hybrid_percentage",
            "skill_percentage",
            "tfidf_percentage",
            "semantic_percentage",
            "demand_percentage",
            "education_percentage"
        ]
    ].head(20)
)

Final Top-5 rows: 315
Final Top-1 rows: 63

Example final recommendations:


,candidate_id,final_hybrid_rank,selected_occupation,final_hybrid_percentage,skill_percentage,tfidf_percentage,semantic_percentage,demand_percentage,education_percentage
0,Candidate_001,1,Software Developer,84.58,100.00,39.58,100.00,22.91,100.0
1,Candidate_001,2,Information Technology (IT) Analyst,65.30,69.68,100.00,73.10,26.64,100.0
2,Candidate_001,3,Bookkeeper,60.34,61.28,23.08,65.46,29.92,100.0
3,Candidate_001,4,Secondary School Teacher,59.20,62.29,5.54,57.50,36.36,100.0
4,Candidate_001,5,Office Administrator,46.58,43.62,3.15,59.26,52.86,0.0
15,Candidate_002,1,Software Developer,78.25,81.90,100.00,100.00,22.91,100.0
16,Candidate_002,2,Office Manager,67.13,100.00,22.68,33.01,52.86,100.0
17,Candidate_002,3,Information Technology (IT) Analyst,63.42,61.55,62.90,75.84,26.64,100.0
18,Candidate_002,4,Office Administrator,62.21,81.98,8.74,36.99,52.86,100.0
19,Candidate_002,5,Secondary School Teacher,61.01,78.23,24.15,46.73,36.36,100.0


In [57]:
# ============================================================
# 16. BASELINE VS FINAL HYBRID
# ============================================================

# Week 5 baseline
baseline = candidate_occupation_scores.copy()

# Resolve one Top-1 row per candidate using score + rank.
baseline = baseline.sort_values(
    ["candidate_id", "occupation_rank", "recommendation_score"],
    ascending=[True, True, False]
)

baseline_top1 = (
    baseline.groupby("candidate_id", as_index=False)
    .first()
)[
    ["candidate_id", "selected_occupation", "recommendation_percentage"]
].rename(
    columns={
        "selected_occupation": "baseline_occupation",
        "recommendation_percentage": "baseline_percentage"
    }
)

hybrid_top1 = final_top1[
    ["candidate_id", "selected_occupation", "final_hybrid_percentage"]
].rename(
    columns={
        "selected_occupation": "hybrid_occupation",
        "final_hybrid_percentage": "hybrid_percentage"
    }
)

baseline_vs_hybrid = baseline_top1.merge(
    hybrid_top1, on="candidate_id", how="inner"
)

baseline_vs_hybrid["recommendation_changed"] = (
    baseline_vs_hybrid["baseline_occupation"]
    != baseline_vs_hybrid["hybrid_occupation"]
)

changed_count = int(baseline_vs_hybrid["recommendation_changed"].sum())
unchanged_count = int(len(baseline_vs_hybrid) - changed_count)
change_percentage = changed_count / len(baseline_vs_hybrid) * 100

print("Baseline vs Final Hybrid")
print("Candidates:", len(baseline_vs_hybrid))
print("Changed:", changed_count)
print("Unchanged:", unchanged_count)
print("Change percentage:", round(change_percentage, 2), "%")

assert len(baseline_vs_hybrid) == 63
assert baseline_vs_hybrid["candidate_id"].nunique() == 63

display(
    baseline_vs_hybrid[
        baseline_vs_hybrid["recommendation_changed"]
    ].head(20)
)

Baseline vs Final Hybrid
Candidates: 63
Changed: 24
Unchanged: 39
Change percentage: 38.1 %


,candidate_id,baseline_occupation,baseline_percentage,hybrid_occupation,hybrid_percentage,recommendation_changed
1,Candidate_002,Office Manager,54.676259,Software Developer,78.25,True
6,Candidate_007,Administrative Assistant,100.000000,Food Service Supervisor,74.31,True
9,Candidate_010,Office Manager,97.122302,Food Service Supervisor,72.11,True
14,Candidate_015,Administrative Assistant,100.000000,Food Service Supervisor,75.75,True
17,Candidate_018,Administrative Assistant,100.000000,Food Service Supervisor,78.54,True
19,Candidate_020,Bookkeeper,62.854108,Software Developer,83.13,True
20,Candidate_021,Bookkeeper,62.854108,Software Developer,83.13,True
21,Candidate_022,Office Manager,97.122302,Food Service Supervisor,72.81,True
22,Candidate_023,Office Manager,54.676259,Software Developer,78.25,True
23,Candidate_024,Office Manager,74.100719,Software Developer,70.96,True


In [59]:
# ============================================================
# 17. PRELIMINARY RECOMMENDATION ERROR ANALYSIS
# ============================================================

# This is recommendation error analysis, not classification error analysis.

error_rows = components.copy()

# 1. Low final score
error_rows["low_final_score_flag"] = (
    error_rows["final_hybrid_score"] <=
    error_rows.groupby("candidate_id")["final_hybrid_score"].transform(
        lambda x: x.quantile(0.25)
    )
)

# 2. Large skill-vs-semantic disagreement
error_rows["skill_semantic_gap"] = (
    error_rows["skill_normalized_score"]
    - error_rows["semantic_normalized_score"]
).abs()

error_rows["large_model_disagreement_flag"] = (
    error_rows["skill_semantic_gap"] >=
    error_rows.groupby("candidate_id")["skill_semantic_gap"].transform(
        lambda x: x.quantile(0.75)
    )
)

# 3. High demand but weak candidate match
error_rows["candidate_match_score"] = (
    0.5 * error_rows["skill_normalized_score"] +
    0.5 * error_rows["semantic_normalized_score"]
)

error_rows["demand_candidate_mismatch"] = (
    (error_rows["demand_score"] >= 0.75) &
    (error_rows["candidate_match_score"] <= 0.25)
)

# 4. High education alignment but weak candidate match
error_rows["education_candidate_mismatch"] = (
    (error_rows["education_score"] >= 0.75) &
    (error_rows["candidate_match_score"] <= 0.25)
)

# 5. Final recommendation driven strongly by demand
error_rows["demand_lift"] = (
    0.20 * error_rows["demand_score"]
)
error_rows["education_lift"] = (
    0.10 * error_rows["education_score"]
)

error_analysis = error_rows[
    error_rows[
        [
            "low_final_score_flag",
            "large_model_disagreement_flag",
            "demand_candidate_mismatch",
            "education_candidate_mismatch"
        ]
    ].any(axis=1)
].copy()

print("Potential recommendation-review cases:", len(error_analysis))

display(
    error_analysis.sort_values(
        ["candidate_id", "final_hybrid_rank"]
    )[
        [
            "candidate_id",
            "selected_occupation",
            "final_hybrid_rank",
            "final_hybrid_percentage",
            "candidate_match_score",
            "skill_semantic_gap",
            "demand_percentage",
            "education_percentage",
            "low_final_score_flag",
            "large_model_disagreement_flag",
            "demand_candidate_mismatch",
            "education_candidate_mismatch"
        ]
    ].head(30)
)

Potential recommendation-review cases: 512


,candidate_id,selected_occupation,final_hybrid_rank,final_hybrid_percentage,candidate_match_score,skill_semantic_gap,demand_percentage,education_percentage,low_final_score_flag,large_model_disagreement_flag,demand_candidate_mismatch,education_candidate_mismatch
6,Candidate_001,Licensed Practical Nurse (L.P.N.),7,41.04,0.356558,0.317689,30.39,100.0,False,True,False,False
8,Candidate_001,"Driver, Truck",9,34.68,0.241659,0.023046,88.80,0.0,False,False,True,False
9,Candidate_001,Food Service Supervisor,10,34.48,0.297266,0.521808,68.33,0.0,False,True,False,False
10,Candidate_001,Continuing Care Assistant,11,34.26,0.363808,0.339013,43.96,0.0,False,True,False,False
11,Candidate_001,Administrative Assistant,12,33.64,0.355364,0.710727,43.85,0.0,True,True,False,False
12,Candidate_001,Inside Sales Representative,13,31.73,0.206258,0.121874,86.44,0.0,True,False,True,False
13,Candidate_001,Retail Sales Associate,14,31.06,0.196744,0.102847,86.44,0.0,True,False,True,False
14,Candidate_001,Delivery Driver,15,12.45,0.071976,0.143952,37.05,0.0,True,False,False,False
16,Candidate_002,Office Manager,2,67.13,0.665047,0.669906,52.86,100.0,False,True,False,False
18,Candidate_002,Office Administrator,4,62.21,0.594850,0.449886,52.86,100.0,False,True,False,False


In [61]:
# ============================================================
# 18. MODEL RANKING COMPARISON
# ============================================================

ranking_comparison = pd.DataFrame({
    "candidate_id": final_top1["candidate_id"].values,
    "keyword_rank": keyword_recommendations[
        keyword_recommendations["keyword_rank"] == 1
    ].sort_values("candidate_id")["keyword_rank"].values,
    "tfidf_rank": tfidf_recommendations[
        tfidf_recommendations["tfidf_rank"] == 1
    ].sort_values("candidate_id")["tfidf_rank"].values,
    "semantic_rank": semantic_recommendations[
        semantic_recommendations["semantic_rank"] == 1
    ].sort_values("candidate_id")["semantic_rank"].values,
    "final_hybrid_rank": final_top1.sort_values("candidate_id")[
        "final_hybrid_rank"
    ].values
})

print("All model top-1 ranking coverage validated.")
assert ranking_comparison.shape[0] == 63

All model top-1 ranking coverage validated.


In [67]:
# ============================================================
# 19. OCCUPATION DISTRIBUTION AND TRANSITIONS
# ============================================================

baseline_distribution = (
    baseline_vs_hybrid["baseline_occupation"]
    .value_counts()
    .rename("baseline_candidate_count")
)

hybrid_distribution = (
    baseline_vs_hybrid["hybrid_occupation"]
    .value_counts()
    .rename("hybrid_candidate_count")
)

occupation_distribution = (
    pd.concat([baseline_distribution, hybrid_distribution], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(columns={"index": "selected_occupation"})
)

occupation_distribution["recommendation_difference"] = (
    occupation_distribution["hybrid_candidate_count"]
    - occupation_distribution["baseline_candidate_count"]
)

transitions = baseline_vs_hybrid[
    baseline_vs_hybrid["recommendation_changed"]
].copy()

transitions["transition"] = (
    transitions["baseline_occupation"]
    + " → " +
    transitions["hybrid_occupation"]
)

transition_summary = (
    transitions["transition"]
    .value_counts()
    .rename_axis("recommendation_transition")
    .reset_index(name="candidate_count")
)

if len(transitions) > 0:
    transition_summary["percentage_of_changed"] = (
        transition_summary["candidate_count"]
        / len(transitions) * 100
    ).round(2)
else:
    transition_summary["percentage_of_changed"] = 0

print("Occupation distribution:")
display(occupation_distribution.sort_values(
    "hybrid_candidate_count", ascending=False
))

print("\nTransition summary:")
display(transition_summary)

Occupation distribution:


,selected_occupation,baseline_candidate_count,hybrid_candidate_count,recommendation_difference
0,Software Developer,22,35,13
1,Administrative Assistant,20,13,-7
5,Food Service Supervisor,0,11,11
2,Office Manager,15,4,-11
3,Bookkeeper,5,0,-5
4,"Driver, Truck",1,0,-1



Transition summary:


,recommendation_transition,candidate_count,percentage_of_changed
0,Office Manager → Software Developer,6,25.00
1,Administrative Assistant → Food Service Superv...,6,25.00
2,Office Manager → Food Service Supervisor,5,20.83
3,Bookkeeper → Software Developer,5,20.83
4,"Driver, Truck → Software Developer",1,4.17
5,Administrative Assistant → Software Developer,1,4.17


In [69]:
# ============================================================
# 20. SAVE ALL WEEK 6 DELIVERABLES
# ============================================================

files_to_save = {
    "week6_keyword_baseline_recommendations.csv": keyword_recommendations,
    "week6_tfidf_recommendations.csv": tfidf_recommendations,
    "week6_semantic_recommendations.csv": semantic_recommendations,
    "week6_education_alignment_scores.csv": education_alignment,
    "week6_all_model_components.csv": components,
    "week6_final_hybrid_recommendations.csv": components,
    "week6_final_top5_recommendations.csv": final_top5,
    "week6_final_top1_recommendations.csv": final_top1,
    "week6_model_comparison.csv": model_summary,
    "week6_baseline_vs_hybrid.csv": baseline_vs_hybrid,
    "week6_error_analysis.csv": error_analysis,
    "week6_occupation_distribution_comparison.csv": occupation_distribution,
    "week6_recommendation_transitions.csv": transition_summary
}

for filename, dataframe in files_to_save.items():
    path = os.path.join(WEEK6_PATH, filename)
    dataframe.to_csv(path, index=False)

print("Saved files:")
for filename in files_to_save:
    print("-", filename)

Saved files:
- week6_keyword_baseline_recommendations.csv
- week6_tfidf_recommendations.csv
- week6_semantic_recommendations.csv
- week6_education_alignment_scores.csv
- week6_all_model_components.csv
- week6_final_hybrid_recommendations.csv
- week6_final_top5_recommendations.csv
- week6_final_top1_recommendations.csv
- week6_model_comparison.csv
- week6_baseline_vs_hybrid.csv
- week6_error_analysis.csv
- week6_occupation_distribution_comparison.csv
- week6_recommendation_transitions.csv


In [71]:
# ============================================================
# 21. FINAL 100% WEEK 6 VALIDATION CHECK
# ============================================================

expected_pairs = (
    candidate_features["candidate_id"].nunique()
    * occupation_documents["selected_occupation"].nunique()
)

checks = {
    "63 candidates": candidate_features["candidate_id"].nunique() == 63,
    "15 occupations": occupation_documents["selected_occupation"].nunique() == 15,
    "945 candidate-occupation pairs": len(components) == expected_pairs == 945,
    "No duplicate pairs": components.duplicated(
        ["candidate_id", "selected_occupation"]
    ).sum() == 0,
    "No missing final scores": components["final_hybrid_score"].isna().sum() == 0,
    "Final scores between 0 and 1": components["final_hybrid_score"].between(0, 1).all(),
    "63 Top-1 recommendations": len(final_top1) == 63,
    "315 Top-5 recommendations": len(final_top5) == 315,
    "All candidates have Top-1": final_top1["candidate_id"].nunique() == 63,
    "Demand covers 15 occupations": occupation_demand["selected_occupation"].nunique() == 15,
    "Education covers all 945 pairs": len(education_alignment) == 945,
    "Weights sum to 1": abs(
        SKILL_WEIGHT + SEMANTIC_WEIGHT +
        DEMAND_WEIGHT + EDUCATION_WEIGHT - 1
    ) < 1e-9
}

print("=" * 80)
print("WEEK 6 — FINAL VALIDATION")
print("=" * 80)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

assert all(checks.values())

print("\n" + "=" * 80)
print("WEEK 6 COMPLETE — ALL REQUIRED TECHNICAL CHECKS PASSED")
print("=" * 80)

print("\nFinal methodology:")
print("35% O*NET/Week-5 skill match")
print("35% Sentence-Transformer semantic similarity")
print("20% Canadian labour demand")
print("10% CIP-based education alignment")

print("\nFinal outputs:", len(files_to_save))
print("Candidate coverage:", final_top1["candidate_id"].nunique())
print("Candidate-occupation combinations:", len(components))

WEEK 6 — FINAL VALIDATION
PASS — 63 candidates
PASS — 15 occupations
PASS — 945 candidate-occupation pairs
PASS — No duplicate pairs
PASS — No missing final scores
PASS — Final scores between 0 and 1
PASS — 63 Top-1 recommendations
PASS — 315 Top-5 recommendations
PASS — All candidates have Top-1
PASS — Demand covers 15 occupations
PASS — Education covers all 945 pairs
PASS — Weights sum to 1

WEEK 6 COMPLETE — ALL REQUIRED TECHNICAL CHECKS PASSED

Final methodology:
35% O*NET/Week-5 skill match
35% Sentence-Transformer semantic similarity
20% Canadian labour demand
10% CIP-based education alignment

Final outputs: 13
Candidate coverage: 63
Candidate-occupation combinations: 945


In [73]:
# ============================================================
# 20. MODEL COMPARISON TABLE
# ============================================================

print("=" * 80)
print("WEEK 6 — RECOMMENDATION MODEL COMPARISON")
print("=" * 80)

model_comparison = pd.DataFrame({
    "Model": [
        "Keyword Baseline",
        "TF-IDF",
        "Sentence-Transformer",
        "Demand-Adjusted",
        "Education-Adjusted",
        "Final Hybrid"
    ],
    
    "Score Column": [
        "baseline_score",
        "tfidf_similarity",
        "semantic_similarity",
        "demand_adjusted_score",
        "education_adjusted_score",
        "final_hybrid_score"
    ],
    
    "Primary Method": [
        "O*NET skill overlap",
        "Text similarity using TF-IDF",
        "Semantic similarity using embeddings",
        "Candidate match + Canadian labour demand",
        "Candidate match + CIP education alignment",
        "Skill + semantic + demand + education"
    ],
    
    "Candidate Coverage": [
        components["candidate_id"].nunique(),
        components["candidate_id"].nunique(),
        components["candidate_id"].nunique(),
        components["candidate_id"].nunique(),
        components["candidate_id"].nunique(),
        components["candidate_id"].nunique()
    ],
    
    "Occupation Coverage": [
        components["selected_occupation"].nunique(),
        components["selected_occupation"].nunique(),
        components["selected_occupation"].nunique(),
        components["selected_occupation"].nunique(),
        components["selected_occupation"].nunique(),
        components["selected_occupation"].nunique()
    ]
})

display(model_comparison)

WEEK 6 — RECOMMENDATION MODEL COMPARISON


,Model,Score Column,Primary Method,Candidate Coverage,Occupation Coverage
0,Keyword Baseline,baseline_score,O*NET skill overlap,63,15
1,TF-IDF,tfidf_similarity,Text similarity using TF-IDF,63,15
2,Sentence-Transformer,semantic_similarity,Semantic similarity using embeddings,63,15
3,Demand-Adjusted,demand_adjusted_score,Candidate match + Canadian labour demand,63,15
4,Education-Adjusted,education_adjusted_score,Candidate match + CIP education alignment,63,15
5,Final Hybrid,final_hybrid_score,Skill + semantic + demand + education,63,15


In [75]:
# ============================================================
# MODEL TOP-1 COMPARISON
# ============================================================

def get_top1_recommendations(data, score_column):

    return (
        data
        .sort_values(
            ["candidate_id", score_column],
            ascending=[True, False]
        )
        .groupby("candidate_id", as_index=False)
        .first()
        [
            [
                "candidate_id",
                "selected_occupation",
                score_column
            ]
        ]
        .rename(
            columns={
                "selected_occupation": "top_occupation"
            }
        )
    )


# ------------------------------------------------------------
# Generate Top-1 recommendations for each model
# ------------------------------------------------------------

baseline_top1 = get_top1_recommendations(
    components,
    "recommendation_score"
)

tfidf_top1 = get_top1_recommendations(
    components,
    "tfidf_normalized_score"
)

semantic_top1 = get_top1_recommendations(
    components,
    "semantic_normalized_score"
)

demand_top1 = get_top1_recommendations(
    components,
    "demand_adjusted_score"
)

education_top1 = get_top1_recommendations(
    components,
    "education_adjusted_score"
)

hybrid_top1 = get_top1_recommendations(
    components,
    "final_hybrid_score"
)


# ------------------------------------------------------------
# Display coverage
# ------------------------------------------------------------

print("=" * 70)
print("TOP-1 RECOMMENDATION COVERAGE BY MODEL")
print("=" * 70)

print(
    "\nKeyword Baseline:",
    baseline_top1["candidate_id"].nunique()
)

print(
    "TF-IDF:",
    tfidf_top1["candidate_id"].nunique()
)

print(
    "Sentence-Transformer:",
    semantic_top1["candidate_id"].nunique()
)

print(
    "Demand-Adjusted:",
    demand_top1["candidate_id"].nunique()
)

print(
    "Education-Adjusted:",
    education_top1["candidate_id"].nunique()
)

print(
    "Final Hybrid:",
    hybrid_top1["candidate_id"].nunique()
)

TOP-1 RECOMMENDATION COVERAGE BY MODEL

Keyword Baseline: 63
TF-IDF: 63
Sentence-Transformer: 63
Demand-Adjusted: 63
Education-Adjusted: 63
Final Hybrid: 63


In [77]:
# ============================================================
# FINAL MODEL COMPARISON SUMMARY
# ============================================================

model_results = pd.DataFrame({

    "Model": [
        "Keyword Baseline",
        "TF-IDF",
        "Sentence-Transformer",
        "Demand-Adjusted",
        "Education-Adjusted",
        "Final Hybrid"
    ],

    "Score Used": [
        "recommendation_score",
        "tfidf_normalized_score",
        "semantic_normalized_score",
        "demand_adjusted_score",
        "education_adjusted_score",
        "final_hybrid_score"
    ],

    "Top-1 Recommendations": [
        len(baseline_top1),
        len(tfidf_top1),
        len(semantic_top1),
        len(demand_top1),
        len(education_top1),
        len(hybrid_top1)
    ],

    "Unique Candidates": [
        baseline_top1["candidate_id"].nunique(),
        tfidf_top1["candidate_id"].nunique(),
        semantic_top1["candidate_id"].nunique(),
        demand_top1["candidate_id"].nunique(),
        education_top1["candidate_id"].nunique(),
        hybrid_top1["candidate_id"].nunique()
    ],

    "Missing Scores": [
        components["recommendation_score"].isna().sum(),
        components["tfidf_normalized_score"].isna().sum(),
        components["semantic_normalized_score"].isna().sum(),
        components["demand_adjusted_score"].isna().sum(),
        components["education_adjusted_score"].isna().sum(),
        components["final_hybrid_score"].isna().sum()
    ]

})

print("=" * 80)
print("WEEK 6 — FINAL MODEL COMPARISON")
print("=" * 80)

display(model_results)

WEEK 6 — FINAL MODEL COMPARISON


,Model,Score Used,Top-1 Recommendations,Unique Candidates,Missing Scores
0,Keyword Baseline,recommendation_score,63,63,0
1,TF-IDF,tfidf_normalized_score,63,63,0
2,Sentence-Transformer,semantic_normalized_score,63,63,0
3,Demand-Adjusted,demand_adjusted_score,63,63,0
4,Education-Adjusted,education_adjusted_score,63,63,0
5,Final Hybrid,final_hybrid_score,63,63,0


## Week 6 deliverables completed

- Keyword baseline
- TF-IDF recommendation model
- Sentence-Transformer semantic model
- Canadian Job Bank demand score
- CIP-based education alignment
- Demand-adjusted recommendation
- Education-adjusted recommendation
- Final hybrid ranking:
  - 35% skill-based candidate–occupation match
  - 35% Sentence-Transformer semantic similarity
  - 20% Canadian labour demand
  - 10% CIP-based education alignment
- Model comparison
- Top-5 and Top-1 recommendations
- Baseline-vs-hybrid analysis
- Recommendation transition analysis
- Preliminary recommendation error analysis
- Validation checks
- CSV exports

